# 8.1 Merging ticker changes
*(For myself I skip this part. Renaming give to much headache, even though it's not that important for short-term signals. Also it does not affect the price itself.)*
This is optional. If you want it ticker-centric or don't want to get a stockanalysis.com subscription, you can just skip this part.

**^^^ Dev: But we have to do it cos we trade multi-week strategies too! And this affects the stocks we trade! FB/META, BITF/KEEL !** 

To get a list of ticker changes, We can loop through all tickers and query <code>Ticker Events</code> but this only works with non-delisted companies. And although you can infer it based on the ticker list by looking at whether the cik or figi has changed, that is very messy. Because a company can stay the same even if the ticker and cik/figi change. I actually did it, and it did found that it did not match the Polygon <code>Ticker Events</code>. Then I stumbled on [stockanalysis.com](https://stockanalysis.com/actions/changes/) where you can find all ticker changes for only 10 bucks a month. The first month is even free. You have to manually download them for each year and put them in the <code>stockanalysis/raw/ticker_changes/</code> folder.

After merging those we will save the result to <code>raw/renamings.csv</code> which will contain the columns <code>['from', 'to', 'now', 'date']</code>.

In [ ]:
from tickers import get_tickers, get_ticker_changes
from times import get_market_dates, get_market_calendar, last_trading_date_before
from datetime import datetime, date, time
from pathlib import Path
import pandas as pd
import shutil
import os
DATA_PATH = "../data/polygon/"
# END_DATE = last_trading_date_before(date(2026, 6, 4)) # till today only or else you will get "out of range" and don't waste loop cycles
# END_DATE will need to support updating everyday so that we only fetch the new adjustments in an idempotent way


In [2]:
# This can be done once and then updates can be done with manually appending the list of ticker changes.
###
# Aggregate the csv's
all_ticker_changes = []
for file in os.listdir(DATA_PATH + "../stockanalysis/raw/ticker_changes/"):
    ticker_changes_year = pd.read_csv(DATA_PATH + "../stockanalysis/raw/ticker_changes/" + file, \
        parse_dates=True, index_col=0, usecols=["Date", "Old", "New"])
    all_ticker_changes.append(ticker_changes_year)

ticker_changes = pd.concat(all_ticker_changes)
ticker_changes.rename(columns={"Old": "from", 
                               "New": "to"}, inplace=True)
ticker_changes.index.names = ['date']
ticker_changes.sort_index(inplace=True)
ticker_changes.to_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")

In [3]:
ticker_changes = pd.read_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")
print(len(ticker_changes))
ticker_changes[ticker_changes['from'] == "FB"]

5897


,date,from,to
4858,2022-06-09,FB,META


# ticker_changes.csv continued and cleaned-up further.
## Do NOT stitch companies that are a "structurally different company/spin-off" after a "corporate actions gap"!
### We must NOT stitch companies that are a "structurally different company" after a "corporate actions gap"! These are NEW companies and MUST be treated as such! DO NOT STITCH THEM! These gaps are "structurally different companies" and as such we can NOT use their histories. Treat them as New companies with new fundamentals and corporate structures.
### Incorrectly identified "Gaps" are detrimental to our "Gap", "Momentum" and Trend-following strategies! So pretty much our ENTIRE System! MUST FIX THIS!
For multi‑week swing systems that rely on price continuity (momentum, moving averages, etc.), a gap of this size is damaging. But the damage is caused by the spin‑off itself, not by the stitching algorithm. 

### Incorrect Stitching (UTX --> RTX example) of spin-offs that are a completely new company with a completely new structure!
What actually happened with UTX → RTX
In April 2020, United Technologies (UTX) didn’t simply rename. It spun off Carrier (CARR) and Otis (OTIS) and then merged the remaining business with Raytheon to form RTX.
- Before the event, a UTX share represented ownership in all three businesses (aerospace + Carrier + Otis).
- After the event, an RTX share represented only the aerospace/defense portion. The other two became separate stocks distributed to shareholders.

For UTX/RTX ticker change, the price drop from ~85.95 to ~53.99 is the market’s repricing after the two spin-offs were removed. It is not a split, a dividend, or an exchange‑ratio conversion (we already "adjusted" for those) — it’s a genuine change in what the stock represents. The stockanalysis.com database likely recorded this as a “ticker change” because UTX did disappear and RTX emerged, but the underlying assets are NOT the same. It's a completely new company with a completely new structure!

Other tickers incorrectly show 1000%+ "gaps"! Let's fix this!

# Solution: 
Information to base decisions on:
- Between 2016 and 2026, there has been: 
    - Found 1193 rename events in the stitched database.
    - **Structural gaps (over 20%) found: 139** 
    - **Structural gaps (over 15%) found: 160** (I am going with this. Better to be safe than sorry!) 
    - SOLUTION: 
        - **139 Stocks that - even after being adjusted for Splits and Dividends - had a "structurally different-company gap" over 20%.**
            - WE DON'T WANT A "Corporate Structure Change" TO BE IDENTIFIED AS A GAPPER! 
        - **Do NOT Stitch! Most of these gaps are "structurally different companies" and as such we can NOT use their histories. Treat them as New companies with new fundamentals and corporate structure.**
        - Unoptimal Alternatives:
            - MUST use Volume Filter in our Scanners to filter out genuine gappers from Splits!
            - And for our MAs and lookback-performance calculations, ALL gaps over 10% MUST be accompanied with a proportional increase in Volume! Otherwise, we mark this gap as an "Adjustment" and do NOT include this gap in the calculation! This means companies that issued Splits (all major companies do this) will still be included in our universe, we just calculate their Performance from Volume-backed "hard" data!
            - **We should "clean" this as part of data processing instead of scanning or within strategies to deal with this headache once and not have it eat our mental capacity when developing strategy ideas.**
            - (OPTIONAL, LATER) Use LLMs to tell us whether the new company was genuinely a structurally different company or not.
                - Claude: "You marked this as optional but dismissed it a bit too quickly. For the **139 flagged cases**, an LLM call against the SEC 8-K filing on the change date is extremely reliable. The 8-K will say exactly what happened (merger, spin-off, pure name change). Cost: ~$2-5 in API calls for 139 cases. Return: permanent ground truth labels in your database."
                - Dev: Good news is that most of these are micro-caps that we likely won't be trading. So just focus on the low-hanging fruit for now and do the LLM intergrations later. 

- BTW 5897 is the raw, unfiltered count from stockanalysis since 1990s. 1193 is the number of rename events that actually made it into your stitched, deduplicated, production‑ready ticker database. The difference is intentional and correct — pipeline is doing exactly what it's supposed to do.

#### LLMs all Agree with the above! 
Your thinking is spot‑on, and the decision to **not stitch** those 139 events is the cleanest, most quant‑robust solution for your use case. Let me confirm why, and then I’ll add a minor refinement.
##### Why breaking the chain for gaps >20% is correct
##### 1. Your strategies are purely technical
Momentum, moving averages, and breakout signals assume **price continuity** – a gap that isn’t backed by genuine trading volume is a false signal. A spin‑off/merger creates a one‑time repricing that no moving average can smooth over; it’s a structural break. Feeding such a distorted series into a strategy would:
- Artificially trigger false breakout/breakdown signals on the stitch date.
- Corrupt lookback calculations (e.g., a 200‑day MA that suddenly jumps 30% overnight).
##### 2. Adjusting for the gap would be equally wrong
You could theoretically scale the old history by the exchange ratio (e.g., UTX → RTX with a ratio of 0.63), but that doesn’t change the fact that the old ticker represented a **different set of assets**. The pre‑event prices simply aren’t comparable to the post‑event entity. Scaling would only mask the discontinuity; it wouldn’t make the series economically continuous.
##### 3. Volume filtering is a hack
Trying to “clean” the gap by requiring a volume spike on the rename date sounds appealing, but:
- A huge volume spike often occurs anyway (e.g., rebalancing by index funds, arbitrage unwinding).
- Even with volume, the gap still represents a corporate action, not a tradeable event for your strategies.
So relying on volume to distinguish “real” gaps from “adjustment” gaps is fragile and adds complexity to every signal. Far better to handle it once in the data layer.
##### Recommended implementation
- **Break the chain** for any rename where the residual gap (after all split/dividend adjustments) exceeds **20%**. You already have the list of 139 events.  
- Keep the old ticker’s data **separate** under its old ID, but mark it as `historical_only` or `structural_break = True` so your scanners never pick it up as a current stock.  
- The new ticker starts with a clean history from its first trading day.  
- Store the fact that a break occurred in your ticker master file (e.g., in `tickers_v5.csv` add a column `break_chain = True`).
This way, your momentum and trend strategies only see stocks that have genuinely continuous price histories.
##### Final word
Your summary is exactly right:
> *“We should clean this as part of data processing instead of within strategies to deal with this headache once.”*

That’s the quant‑dev gold standard. Clean, unambiguous data at the source means all downstream strategies can be simpler and more robust. The 139 break events are a small fraction of your universe (1.4% of rename events), so you lose very little history, but you gain a lot of signal purity.
Proceed with breaking those chains – it’s the right call.


In [ ]:
# Code to detect huge discontinuities
# Here’s a parallelized version using concurrent.futures.ProcessPoolExecutor. 
# It will scan your entire ticker_changes.csv in a fraction of the time by distributing the work across all your CPU cores.
# On a typical modern machine with 8+ cores, this will scan 5,000 changes in under a minute.

from datetime import datetime, time, date
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing

PROCESSED_PATH = os.path.join(DATA_PATH, "processed", "m1")

# Load the final (stitched) ticker list
tickers_v5 = pd.read_csv("../data/tickers_v5.csv")
tickers_v5['tickers_old'] = tickers_v5['tickers_old'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x != '{}' else {}
)

# Flatten all rename events: each is (old_ticker, new_ticker, change_date, current_id)
events = []
for _, row in tickers_v5.iterrows():
    ticker_current = row['ticker']
    tid = row['ID']
    for chg_date_str, old_ticker in row['tickers_old'].items():
        change_date = date.fromisoformat(chg_date_str)
        events.append((old_ticker, ticker_current, change_date, tid))

print(f"Found {len(events)} rename events in the stitched database.")

# Threshold for flagging (now in percentage points, e.g., 20 means ±20%)
GAP_THRESHOLD = 15.0

def check_merged_file(args):
    old_ticker, new_ticker, change_date, tid = args
    parquet_path = os.path.join(PROCESSED_PATH, f"{tid}.parquet")
    if not os.path.exists(parquet_path):
        return None

    try:
        # Read only the bars around the change date to save memory
        # We'll load a range: one day before and one day after
        start_dt = datetime.combine(change_date, time(0)) - pd.Timedelta(days=1)
        end_dt   = datetime.combine(change_date, time(23,59)) + pd.Timedelta(days=1)
        df = pd.read_parquet(parquet_path,
                             filters=[("datetime", ">=", start_dt),
                                      ("datetime", "<=", end_dt)])
        if df.empty:
            return None

        # Ensure index is datetime
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        # The old ticker's last bar: the last bar before change_date
        old_part = df[df.index.date < change_date]
        if old_part.empty:
            return None
        old_last = old_part.iloc[-1]
        old_last_close = old_last['close']

        # The new ticker's first tradeable bar on or after change_date
        new_part = df[(df.index.date >= change_date) & (df['tradeable'] == True)]
        if new_part.empty:
            return None
        new_first = new_part.iloc[0]
        new_first_close = new_first['close']

        # Gap after stitch: ((new_close / old_close) - 1) * 100  (percentage)
        gap_pct = ((new_first_close / old_last_close) - 1) * 100

        if abs(gap_pct) > GAP_THRESHOLD:
            return {
                'old_ticker': old_ticker,
                'new_ticker': new_ticker,
                'change_date': change_date,
                'old_last_close': round(old_last_close, 4),
                'new_first_close': round(new_first_close, 4),
                'gap_pct': round(gap_pct, 4),   # now in percent
                'flag': 'structural_gap'
            }
    except Exception:
        return None
    return None

# Run in parallel
max_workers = min(32, (multiprocessing.cpu_count() or 4) * 4)
print(f"Scanning {len(events)} events using {max_workers} threads…")

gaps = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(check_merged_file, ev): ev for ev in events}
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            gaps.append(result)

gaps_df = pd.DataFrame(gaps)
print(f"Structural gaps over {GAP_THRESHOLD}% found: {len(gaps_df)}")
if not gaps_df.empty:
    gaps_df['abs_gap'] = gaps_df['gap_pct'].abs()
    top = gaps_df.sort_values('abs_gap', ascending=False).head(10)
    print("\nTop 10 largest residual gaps (in %):")
    print(top[['old_ticker', 'new_ticker', 'change_date', 'gap_pct']])
    out_path = os.path.join(DATA_PATH, "processed", "structural_gaps_merged.csv")
    gaps_df.to_csv(out_path, index=False)
    print(f"Full list saved to {out_path}")
else:
    print("No structural gaps detected.")

Found 1193 rename events in the stitched database.
Scanning 1193 events using 32 threads…
Structural gaps over 15.0% found: 160

Top 100 largest residual gaps (in %):
    old_ticker new_ticker change_date      gap_pct
108       HUGE       QNTM  2024-08-15  112291.3043
29        SESN       CARM  2023-03-08  -29046.2810
78         DPS        KDP  2018-07-10   -7229.9238
135       PRTO       TARA  2020-01-10    5836.7978
67        KTRA       HURA  2024-10-18    5766.6667
103       TYDE       ORBS  2023-04-04    4900.0000
86        HLBZ       MCOM  2023-03-31    4607.5209
42        MGTA       DNTH  2023-09-12    3034.3284
2         IDRA       ACGN  2023-01-18    2797.6190
157        ADD        ZNB  2025-08-22    2761.9529
Full list saved to ../data/polygon/processed/structural_gaps_merged.csv


In [ ]:
bottom = gaps_df.sort_values('abs_gap', ascending=False).tail(10)
print(bottom[['old_ticker', 'new_ticker', 'change_date', 'gap_pct']])


    old_ticker new_ticker change_date  gap_pct
17        JCIC       BAER  2023-01-25  18.2692
11        DTOC       AONC  2023-09-21  18.1364
20        AVAC       BENF  2023-06-08  17.8904
53        FBHS       FBIN  2022-12-15 -17.5283
125        DTC       SBDS  2025-07-24  17.5060
128         SV        SMR  2022-05-03  17.0732
106        BFT       PSFE  2021-03-31  16.9701
45        LWAC       EFTR  2021-08-26  16.8160
88        MYTE       LUXE  2025-05-01  16.4921
23        SBEA       BRCC  2022-02-10  16.1290


### Code to exclude structurally different Spin-offs from getting stitched!

In [6]:
import shutil

# Define paths matching your notebook structure
CHANGES_CSV = DATA_PATH + "../stockanalysis/ticker_changes.csv"
BACKUP_CSV = DATA_PATH + "../stockanalysis/ticker_changes_backup.csv"
GAPS_CSV = os.path.join(DATA_PATH, "processed", "structural_gaps_merged.csv")

# 1. Create a backup of the original file if it doesn't already exist
if os.path.exists(CHANGES_CSV) and not os.path.exists(BACKUP_CSV):
    shutil.copy2(CHANGES_CSV, BACKUP_CSV)
    print(f"Created raw backup of ticker changes at: {BACKUP_CSV}")
elif os.path.exists(BACKUP_CSV):
    print(f"Backup already exists at: {BACKUP_CSV}. Using existing backup for safe re-runs.")

# 2. Load the original aggregated ticker changes (Prefer backup to allow idempotent re-runs)
if os.path.exists(BACKUP_CSV):
    ticker_changes = pd.read_csv(BACKUP_CSV)
else:
    ticker_changes = pd.read_csv(CHANGES_CSV)

# If 'date' was saved as the index, reset it so we can manipulate it easily
if 'date' not in ticker_changes.columns and ticker_changes.index.name == 'date':
    ticker_changes = ticker_changes.reset_index()

print(f"Original ticker_changes count: {len(ticker_changes)}")

# 3. Load the identified structural gaps and filter
if os.path.exists(GAPS_CSV):
    structural_gaps = pd.read_csv(GAPS_CSV)
    
    # Create a set of (old, new) tuples for O(1) lookups and rock-solid matching
    gaps_to_exclude = set(zip(structural_gaps['old_ticker'], structural_gaps['new_ticker']))
    
    print(f"Found {len(gaps_to_exclude)} structurally different spin-offs/mergers to exclude.")
    
    # Filter out the structural gaps from our stitching pipeline
    mask = ticker_changes.apply(
        lambda row: (row['from'], row['to']) not in gaps_to_exclude, 
        axis=1
    )
    ticker_changes_cleaned = ticker_changes[mask].copy()
    
    print(f"Cleaned ticker_changes count: {len(ticker_changes_cleaned)}")
    
    # 4. Set index back to 'date' to maintain structural parity with original df
    if 'date' in ticker_changes_cleaned.columns:
        ticker_changes_cleaned.set_index('date', inplace=True)
        
    # 5. Overwrite the original CSV file!
    ticker_changes_cleaned.to_csv(CHANGES_CSV)
    print(f"Successfully OVERWRITTEN {CHANGES_CSV} with structurally sound tickers only.")
    
    # Display a sample of what got excluded for sanity check
    excluded_df = ticker_changes[~mask]
    print("\nSample of excluded structural breaks:")
    display(excluded_df.head(10))
    
else:
    print(f"File not found: {GAPS_CSV}. Please ensure the gap detection cell ran successfully.")

Created raw backup of ticker changes at: ../data/polygon/../stockanalysis/ticker_changes_backup.csv
Original ticker_changes count: 5897
Found 160 structurally different spin-offs/mergers to exclude.
Cleaned ticker_changes count: 5736
Successfully OVERWRITTEN ../data/polygon/../stockanalysis/ticker_changes.csv with structurally sound tickers only.

Sample of excluded structural breaks:


,date,from,to
3222,2017-10-18,FSFR,OCSI
3240,2017-11-14,HOTR,SONN
3246,2017-11-16,ADHD,ARCT
3258,2017-12-01,PBMD,IMMP
3276,2018-01-05,ITEK,RCKT
3283,2018-01-18,NEOT,EVFM
3323,2018-03-27,ENRJ,UAVS
3380,2018-07-10,DPS,KDP
3441,2018-10-11,WMIH,COOP
3443,2018-10-16,VSAR,ARAV


### Minor issue - Inferred Renamings (AAII went delinquent and then the ticker became AAIIE) (Optional since "delinquent" stocks will never hit our scans)
Seems there are special conditions, such as 'delinquent', which adds an extra letter at the end of the ticker. E.g. AAII went delinquent and then the ticker became AAIIE. However these are not real ticker changes so it is not contained in the stockanalysis database. However we can easily infer it from our own ticker list: if the dates are consecutive and an extra letter is added, we can infer the ticker change. We will save this to <code>raw/inferred_renamings.csv.</code>

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = "../data/polygon/"
RAW       = DATA_PATH + "raw"
PROCESSED = DATA_PATH + "processed"

# ── Status suffix allow-list ──────────────────────────────────────────────────
# Restricting to exchange-assigned status letters eliminates false positives
# (two unrelated companies that happen to satisfy the date+prefix rule).
#   E  SEC-reporting delinquent
#   Q  bankruptcy / liquidation proceedings
#   D  new issue (when-issued / distribution)
# https://chatgpt.com/share/6a327201-6d30-83eb-977a-f0cae5c483ea - no need for other Suffixes like D or V since those have false positives and I want "Precision over Recall" - The Quant Golden Rule. 
# And besides, if a ticker gets to that stage, it's not gonnamake our scanner. Let's not waste time on these. 
# Out of 11k clean tickers, we only saw 1 D and 1 C and those went bankrupt within the year. 
KNOWN_SUFFIXES = {"E", "Q"}

# ── Load ticker list ──────────────────────────────────────────────────────────
tickers_v4 = pd.read_csv(
    DATA_PATH + "../tickers_v4.csv",
    parse_dates=["start_date", "end_date", "start_data", "end_data"],
)
tickers_v4 = tickers_v4[["ticker", "start_date", "end_date"]].copy()

# ── Build trading calendar from the dates in the file ─────────────────────────
# Every date in the file is a valid trading day, so the union of start/end
# dates gives us a complete-enough calendar for the next-day lookup.
all_dates    = pd.concat([tickers_v4["start_date"], tickers_v4["end_date"]]).dropna().unique()
trading_days = sorted(all_dates)

next_trading_day_map = {
    d: trading_days[i + 1]
    for i, d in enumerate(trading_days[:-1])
}

# ── Fast lookup: start_date → set of tickers starting on that day ─────────────
start_date_index = (
    tickers_v4
    .groupby("start_date")["ticker"]
    .apply(set)
    .to_dict()
)

# ── Detect status-suffix renamings ────────────────────────────────────────────
records = []

for row in tickers_v4.itertuples(index=False):   # itertuples: no per-row Series overhead
    old_ticker = row.ticker
    last_date  = row.end_date

    next_day = next_trading_day_map.get(last_date)
    if next_day is None:
        continue

    for new_ticker in start_date_index.get(next_day, set()):
        # Strip the candidate suffix first; check allow-list before the
        # (slightly more expensive) full string comparison.
        if len(new_ticker) != len(old_ticker) + 1:
            continue
        suffix = new_ticker[-1]
        if suffix not in KNOWN_SUFFIXES:          # O(1) set lookup — cheap gate
            continue
        if new_ticker[:-1] != old_ticker:         # exact root match
            continue

        records.append({
            "date":         next_day,
            "ticker_old":   old_ticker,
            "ticker_new":   new_ticker,
            "suffix_added": suffix,
        })

inferred_df = (
    pd.DataFrame(records, columns=["date", "ticker_old", "ticker_new", "suffix_added"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Found {len(inferred_df)} inferred renamings")
print(inferred_df)

if not inferred_df.empty:
    inferred_df.to_csv(PROCESSED / "inferred_renamings.csv", index=False)


Found 0 inferred renamings
Empty DataFrame
Columns: [date, ticker_old, ticker_new, suffix_added]
Index: []


# Stitching
There are a whopping 4347 ticker changes from 2003 to now that we have to take care of. But at least this was very easy to get.

Now we will use them to merge our data. We have to be aware that it is possible for a ticker to used multiple times, so the <code>ticker_changes.csv</code> may contain multiple of the same tickers in the 'from' and 'to' column. 

After processing the ticker changes we will create a <code>tickers_v5.csv</code> which will be our definitive ticker list. This contains a column 'tickers_old', which will containa list of (date_of_change, ticker) pairs. So if A changes to B on day 2, and B changes to C on day 5, tickers_old for D will contain [[2, A], [5, B]].

The process will be as follows:
* As long as we have ticker changes to process
    * Loop through <code>tickers_v4.csv</code>.
        * Get the next trading date after 'end_date_data'.
        * Search in <code>tickers_changes.csv</code> if there is a ticker change on this date.
        * If it does:
            * The stock data will be merged.
            * In <code>tickers_v4.csv</code> we will change "ticker" to the new ticker and add a list [date, ticker] to "tickers_old".
            * All other rows will be merged such as "start_date". For identifiers such as FIGI we will take the last available value. For the ID we will keep the original. If we do not do this, we might run into problems with identical IDs.
            * The row of the old ticker will be deleted
            * **We need to restart the loop!** If we don't the following can happen: Let's assume that a ticker was renamed from A -> B -> C -> D but that the order in which it appears in our ticker list is C, D, A, B. Using our loop, C gets merged with D. Then the loop checks D, which has no renamings. Then A gets merged with B. Then B gets merged with C, however that is incorrect! B should be merged with the new D, which contains C. Any double+ renamings have the risk of being in the 'wrong order'.
                * For this to work, the ticker list must be sorted on end_date.
            * Of course we must not forget that there can be adjustments on day 1 of the ticker change. There should be laws to prohibit this.

Note: if a ticker A goes OTC and then comes back and changes to B, then we will have two files: one of the A before OTC and the A+B after OTC named B. This is as intended.

#### Bug Fix

Here’s what happens:
```python
market_hours = get_market_calendar()
market_hours = market_hours[['regular_close']]

cum_div_date = end_data_from
# cum_div_time = market_hours.loc[cum_div_date][0]
# BUG FIX = market_hours after the subset is a DataFrame with one column: 'regular_close'. 
cum_div_time = market_hours.loc[cum_div_date, 'regular_close']
```
market_hours after the subset is a DataFrame with one column: 'regular_close'.

When you do market_hours.loc[cum_div_date], you get a Series whose index is ['regular_close'] (the column names), not integer positions.

Then [0] tries to look up label 0 in that Series’ index — which doesn’t exist → KeyError: 0.

The earlier part of your code already uses the correct pattern for the same kind of lookup:
```python
calendar = get_market_calendar('datetime')
start_data_to_dt = calendar.loc[start_data_to, 'regular_close']   # ✅ works
```

So just copy that style:
```python
cum_div_time = market_hours.loc[cum_div_date, 'regular_close']
```

#### Bug Fix
```python
# split_amount = split['amount'][0]
split_amount = split['amount'].iloc[0]   # fixed

# dividend_amount = dividend['amount'][0]
dividend_amount = dividend['amount'].iloc[0]   # fixed
```

Why does [0] fail?
Because adjustments has a datetime index (you set index_col=0 on read, then convert to date). So dividend['amount'] is a Series indexed by date, not by integer position. [0] tries to look up the label 0 in that date index → KeyError.

Fix both lines (for split and dividend) by using positional indexing. That will get the first value regardless of the index type.

### Dev's Extremely Important Improvement

```python
# Get ticker changes 
# BUG_FIX
# Try to find the change on the old ticker's last day first, then on the new ticker's first day. 
# This fixes the issue where sometimes stockanalysis records change date as "first day of the new ticker" 
# and sometimes as "last day of the old ticker". 
# Example: ticker changes like UTX --> RTX did NOT get stitched cos of this. So we apply the following Fix. 
change = ticker_changes[(ticker_changes.index == end_data_from) & (ticker_changes['from'] == ticker_from)]
if change.empty:
    change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
# print(change)
# change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
```

In [5]:
tickers_v4 = get_tickers(v=4)
# QUICK BUG FIX, NEED TO REWRITE CODE TO MAKE IT CHRONOLOGICAL
market_dates = get_market_dates()
ticker_changes = get_ticker_changes()

END_DATE = market_dates[-1] # to not get "out of range" errors

# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
# These (e.g. AAII → AAIIE) are not in stockanalysis.com but were inferred
# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
try:
    inferred_path = PROCESSED + "inferred_renamings.csv"
    if os.path.exists(inferred_path):
        inferred = pd.read_csv(inferred_path, parse_dates=["date"])
        inferred["date"] = pd.to_datetime(inferred["date"]).dt.date

        tc_flat  = ticker_changes.reset_index()
        date_col = tc_flat.columns[0]

        inf_flat = (
            inferred[["date", "ticker_old", "ticker_new"]]
            .rename(columns={"date": date_col, "ticker_old": "from", "ticker_new": "to"})
        )

        n_before = len(ticker_changes)
        ticker_changes = (
            pd.concat([tc_flat, inf_flat], ignore_index=True)
            .drop_duplicates(subset=[date_col, "from"], keep="first")
            .set_index(date_col)
            .sort_index()
        )
        print(
            f"ticker_changes augmented: {n_before} → {len(ticker_changes)} entries "
            f"(+{len(inf_flat)} inferred suffix renames)"
        )
except Exception as e:
    print(f"[WARN] Inferred renamings augmentation skipped: {e}")
    print("[WARN] Main loop will proceed with unaugmented ticker_changes")

tickers_v4.insert(loc = 2, column = 'tickers_old', value = [{} for _ in range(len(tickers_v4))])

while True:
    tickers_v4 = tickers_v4.sort_values(by='end_data').reset_index(drop=True)

    # tickers_v4 gets smaller by 1 element every time we run this loop.
    for index_from, row_from in tickers_v4.copy().iterrows():
        # Get values
        type_from = row_from['type']
        if type_from == "INDEX":
            continue
        id_from = row_from['ID']
        ticker_from = row_from['ticker']
        start_date_from = row_from['start_date']
        end_date_from = row_from['end_date']
        start_data_from = row_from['start_data']
        end_data_from = row_from['end_data']

        if end_data_from == END_DATE:
            continue

        # ── Safety checks ──────────────────────────────────────────────────────
        if pd.isna(end_data_from):
            print(f"  -> NaT detected! Skipping this ticker.")
            print(f"[WARN] index: {index_from}, ticker: {ticker_from}, "
                        f"end_data_from: {end_data_from} (type: {type(end_data_from)})")
            continue

        start_data_to = market_dates[market_dates.index(end_data_from) + 1]

        # Get ticker changes 
        # BUG_FIX
        # Try to find the change on the old ticker's last day first, then on the new ticker's first day. 
        # This fixes the issue where sometimes stockanalysis records change date as "first day of the new ticker" 
        # and sometimes as "last day of the old ticker". 
        # Example: ticker changes like UTX --> RTX did NOT get stitched cos of this. So we apply the following Fix. 
        change = ticker_changes[(ticker_changes.index == end_data_from) & (ticker_changes['from'] == ticker_from)]
        if change.empty:
            change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
        # print(change)
        # change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]

        if change.empty:
            continue
        elif len(change) > 1:
            raise Exception("Duplicate!")
        ticker_to = change['to'].values[0]

        # Set values of new ticker
        row_to = tickers_v4[(tickers_v4['start_data'] == start_data_to) & (tickers_v4['ticker'] == ticker_to)]
        if row_to.empty:
            continue
        index_to = row_to.index[0]
        id_to = row_to['ID'].values[0]
        tickers_v4.loc[index_to, "tickers_old"][start_data_to.isoformat()] = ticker_from
        tickers_v4.loc[index_to, "start_date"] = start_date_from
        tickers_v4.loc[index_to, "start_data"] = start_data_from

        # --- IDEMPOTENCY CHECK 1 ---
        if not os.path.exists(DATA_PATH + f"processed/m1/{id_from}.parquet"):
            raise FileNotFoundError(f"[ERROR] {DATA_PATH + f"processed/m1/{id_from}.parquet"} is missing! Expected data for {ticker_from}.")

        # Do the actual merging
        from_ = pd.read_parquet(DATA_PATH + f"processed/m1/{id_from}.parquet")
        to = pd.read_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet")

        # When a ticker changes, the adjustments carry over the the old ticker.
            # Get market close minute to calculate the adjustment factor, and adjust 'to'.
            # (Adjustment factor is the same throughout the day, so market close is arbitrarely chosen)
        calendar = get_market_calendar('datetime')
        start_data_to_dt = calendar.loc[start_data_to, 'regular_close']
        start_data_to_dt_bar = to.loc[start_data_to_dt]
        adjustment_factor = start_data_to_dt_bar['close'] / start_data_to_dt_bar['close_original']
        # print(f"Adjustment factor of {adjustment_factor} applied to {id_from}")
        # inside the stitch loop, after reading 'to'
        # print("start_data_to:", start_data_to)
        # print("start_data_to_dt:", start_data_to_dt)
        # bar_used = to.loc[start_data_to_dt]
        # print("bar used for factor:")
        # print(bar_used[['close', 'close_original']])
        # print("ratio:", bar_used['close'] / bar_used['close_original'])
        # any_other_bar = to.loc[to.index.date == start_data_to].iloc[0]
        # print("another bar close/close_original:", any_other_bar['close'] / any_other_bar['close_original'])

        from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
        from_ = round(from_, 4)

        # Because companies like to be annoying, a split/dividend can take place at the same time as a ticker change. We have to account for this. An example is TYDE -> OCTO with a 50:1 reverse split. However this is much easier than 5_process_raw_data.ipynb, because there is at most a single split. This is rare, but there are still a handful of cases.
        if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_to}.csv"):
            adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_to}.csv", parse_dates=True, index_col=0)
            adjustments.index = pd.to_datetime(adjustments.index).date
            adjustments = adjustments[(adjustments.index == start_data_to)]

            # SPLIT ADJUSTMENT
            split = adjustments[adjustments.type == 'SPLIT']
            if len(split) > 0:
                # split_amount = split['amount'][0]
                split_amount = split['amount'].iloc[0]   # fixed

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(split_amount)

            # DIVIDEND ADJUSTMENT - REUN is the only case, not clear what happened there, likely a 'special dividend'
            dividend = adjustments[adjustments.type == 'DIV']
            if len(dividend) > 0:
                print(ticker_to)
                market_hours = get_market_calendar()
                market_hours = market_hours[['regular_close']]

                cum_div_date = end_data_from
                # cum_div_time = market_hours.loc[cum_div_date][0]
                # BUG FIX = market_hours after the subset is a DataFrame with one column: 'regular_close'. 
                cum_div_time = market_hours.loc[cum_div_date, 'regular_close']
                cum_div_datetime = datetime.combine(cum_div_date, cum_div_time)
                cum_div_datetime = (from_[from_.index <= cum_div_datetime].index).max()
                cum_div_close = from_.loc[cum_div_datetime, 'close']
                # dividend_amount = dividend['amount'][0]
                dividend_amount = dividend['amount'].iloc[0]   # fixed

                adjustment_factor_div = 1 - dividend_amount/cum_div_close

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor_div)
            
            # ROUNDING
            if len(split) > 0 or len(dividend) > 0:
                from_ = round(from_, 4)
                from_['turnover'] = from_['turnover'].astype(int)

        # # If on the old ticker, there are divs/splits on start_data_to (start of new ticker), then something is wrong.
        # if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_from}.csv"):
        #     adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_from}.csv", parse_dates=True, index_col=0)
        #     adjustments.index = pd.to_datetime(adjustments.index).date
        #     adjustments = adjustments[(adjustments.index == start_data_to)]
        #     assert len(adjustments) == 0

        # Remove the 'from' ticker, then paste the 'from' and 'to' ticker to m1_renamed for debugging purposes.
        _ = shutil.move(DATA_PATH + f'processed/m1/{id_from}.parquet', DATA_PATH + f'processed/m1_renamed/{id_from}.parquet')
        _ = shutil.copyfile(DATA_PATH + f'processed/m1/{id_to}.parquet', DATA_PATH + f'processed/m1_renamed/{id_to}.parquet')

        pd.concat([from_, to]).to_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet", engine="fastparquet", row_group_offsets=25000)

        tickers_v4.drop(index_from, inplace=True)
        tickers_v4.reset_index(inplace=True, drop=True)
        
        print(f"Ticker change {ticker_from} -> {ticker_to} on {start_data_to} has been processed")
        print(f"{index_from/len(tickers_v4)*100:.1f}% | Length of tickers_v4 is {len(tickers_v4)}")
        break

    # If we have reached the end of the loop, it means we have processed everything. Then we can stop.
    if index_from >= (len(tickers_v4)-1):
        break
    
tickers_v4 = tickers_v4.sort_values(by='ID').reset_index(drop=True)

tickers_v4.to_csv("../data/tickers_v5.csv")

Ticker change GB -> ITGR on 2016-07-01 has been processed
0.1% | Length of tickers_v4 is 11170
Ticker change DNAI -> SRRA on 2017-01-10 has been processed
2.0% | Length of tickers_v4 is 11169
Ticker change MPET -> TELL on 2017-02-10 has been processed
2.2% | Length of tickers_v4 is 11168
Ticker change HCAC -> DSKE on 2017-02-28 has been processed
2.4% | Length of tickers_v4 is 11167
Ticker change SYRG -> SRCI on 2017-03-06 has been processed
2.5% | Length of tickers_v4 is 11166
Ticker change ROIAK -> UONEK on 2017-05-08 has been processed
3.2% | Length of tickers_v4 is 11165
Ticker change BDE -> CLAR on 2017-08-14 has been processed
4.1% | Length of tickers_v4 is 11164
Ticker change URRE -> WWR on 2017-08-21 has been processed
4.2% | Length of tickers_v4 is 11163
Ticker change IFMI -> COHN on 2017-09-05 has been processed
4.3% | Length of tickers_v4 is 11162
Ticker change AUXO -> CTEK on 2017-09-08 has been processed
4.3% | Length of tickers_v4 is 11161
Ticker change BKFS -> BKI on 201

In [6]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

1089


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
5616,META-2022-06-09,META,{'2022-06-09': 'FB'},"Meta Platforms, Inc. Class A Common Stock",True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1326801.0,BBG000MM2P62


Tickers that were renamed multiple times:

In [7]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

95


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
19,AAMI-2025-01-02,AAMI,"{'2025-01-02': 'BSIG', '2018-03-26': 'OMAM'}",Acadian Asset Management Inc.,True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1748824.0,BBG00P2HLNY3
64,ABTC-2025-09-03,ABTC,"{'2025-09-03': 'GRYP', '2024-02-09': 'KERN'}",American Bitcoin Corp. Class A Common Stock,True,2019-06-18,2026-06-03,2019-06-18,2026-06-03,CS,1755953.0,BBG00MS8GZP1


Now we have 5 tickers lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. ETFs added.
4. Data start/end dates added.
5. Renamings merged.
Only the last should be used in backtesting.

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

# Verification

In [8]:
# Perfect!

# Question:
# For UTX → RTX, why is the "close" of 74.2996 on 2020-04-02 19:59:00 lower than "close_original" of 85.95 on that day 
# but "close" of 59.6385 on 2020-04-03 higher than the "close_original" of  53.99 on that day!?
# Adjustment factor of 0.86445108587368 applied to UTX-2016-06-06
# Adjustment factor calculated using real trades EOD:
# 2020-04-03 15:59:00	43.1837	43.4128	43.0670	43.3868	50.190	10146811	202168.0	1192.0	True
# NOT using backfilled bars in the pre-mkt that are filled to just get price continuity somewhat rather than be used for calculations.
# Conclusion: It's all good! Adjustment factor of 0.86445108587368 is the correct adjustment factor. 

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/RTX-2020-04-03.parquet")
# filtered = bars.loc['2020-04-02 19:57:00':'2020-04-03 04:02:00']
filtered = bars.loc['2020-04-02 19:57:00':'2020-04-03 15:59:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2020-04-02 19:57:00,74.2996,74.2996,74.2996,74.2996,85.950,21487,250.0,3.0,True
2020-04-02 19:58:00,74.2996,74.2996,74.2996,74.2996,85.950,0,0.0,0.0,False
2020-04-02 19:59:00,74.2996,74.2996,74.2996,74.2996,85.950,0,0.0,0.0,False
2020-04-03 04:00:00,59.6385,59.6385,59.6385,59.6385,53.990,0,0.0,0.0,False
2020-04-03 04:01:00,59.6385,59.6385,59.6385,59.6385,53.990,0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...
2020-04-03 15:55:00,43.2269,43.3177,43.2139,43.2312,50.010,11073614,221428.0,809.0,True
2020-04-03 15:56:00,43.2399,43.2399,43.2226,43.2312,50.010,5801210,116001.0,610.0,True
2020-04-03 15:57:00,43.2312,43.2917,42.9546,43.2226,50.000,9746950,194939.0,961.0,True


In [9]:
# Perfect!
# Adjustment factor of 40.0 applied to SRNG-2021-04-19 

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/DNA-2021-09-17.parquet")
filtered = bars.loc['2021-09-16 19:57:00':'2021-09-17 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2021-09-16 19:57:00,467.6,468.8,467.6,468.8,11.72,15493,1322.0,8.0,True
2021-09-16 19:58:00,467.6,467.6,466.0,466.0,11.65,4205,361.0,4.0,True
2021-09-16 19:59:00,467.6,470.0,467.6,470.0,11.75,9071,772.0,2.0,True
2021-09-17 04:00:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False
2021-09-17 04:01:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False
2021-09-17 04:02:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False


In [10]:
# Perfect! 
# Adjustment factor of 0.19430549059956972 applied to SNE-2016-06-06

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/SONY-2021-04-01.parquet")
filtered = bars.loc['2021-03-31 19:57:00':'2021-04-01 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2021-03-31 19:57:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-03-31 19:58:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-03-31 19:59:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-04-01 04:00:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False
2021-04-01 04:01:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False
2021-04-01 04:02:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False


In [11]:
# Perfect! 
# 4% Gap up
# Adjustment factor of 1.0 applied to BITF-2021-06-21

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/KEEL-2026-04-06.parquet")
filtered = bars.loc['2026-04-02 19:57:00':'2026-04-06 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2026-04-02 19:57:00,1.9699,1.9699,1.9699,1.9699,1.9699,9772,4961.0,1.0,True
2026-04-02 19:58:00,1.9699,1.9699,1.9699,1.9699,1.9699,0,0.0,0.0,False
2026-04-02 19:59:00,1.9699,1.9699,1.9699,1.9699,1.9699,0,0.0,0.0,False
2026-04-06 04:00:00,2.0500,2.0500,2.0500,2.0500,2.0500,0,0.0,0.0,False
2026-04-06 04:01:00,2.0500,2.0500,2.0500,2.0500,2.0500,0,0.0,0.0,False
2026-04-06 04:02:00,2.0500,2.0500,2.0500,2.0500,2.0500,0,0.0,0.0,False


In [12]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/META-2022-06-09.parquet")
filtered = bars.loc['2022-06-08 19:57:00':'2022-06-09 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2022-06-08 19:57:00,194.6227,194.6227,194.6227,194.6227,196.15,0,0.0,0.0,False
2022-06-08 19:58:00,194.5929,194.5929,194.5929,194.5929,196.12,44323,226.0,11.0,True
2022-06-08 19:59:00,194.7616,194.7616,194.7616,194.7616,196.29,151535,772.0,38.0,True
2022-06-09 04:00:00,194.9303,194.9303,194.9303,194.9303,196.46,0,0.0,0.0,False
2022-06-09 04:01:00,194.9303,194.9303,194.9303,194.9303,196.46,0,0.0,0.0,False
2022-06-09 04:02:00,194.9303,194.9303,194.9303,194.9303,196.46,0,0.0,0.0,False


In [13]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/VSXY-2026-06-02.parquet")
filtered = bars.loc['2026-06-01 19:57:00':'2026-06-02 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2026-06-01 19:57:00,55.2,55.2,55.2,55.2,55.2,0,0.0,0.0,False
2026-06-01 19:58:00,55.2,55.2,55.2,55.2,55.2,0,0.0,0.0,False
2026-06-01 19:59:00,55.2,55.2,55.2,55.2,55.2,0,0.0,0.0,False
2026-06-02 04:00:00,54.3,54.3,54.3,54.3,54.3,5430,100.0,1.0,True
2026-06-02 04:01:00,54.3,54.3,54.3,54.3,54.3,0,0.0,0.0,False
2026-06-02 04:02:00,54.3,54.3,54.3,54.3,54.3,0,0.0,0.0,False


In [14]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/XYZ-2025-01-21.parquet")
filtered = bars.loc['2025-01-17 19:57:00':'2025-01-21 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2025-01-17 19:57:00,87.17,87.17,87.17,87.17,87.17,0,0.0,0.0,False
2025-01-17 19:58:00,87.17,87.17,87.17,87.17,87.17,0,0.0,0.0,False
2025-01-17 19:59:00,87.17,87.17,87.17,87.17,87.17,0,0.0,0.0,False
2025-01-21 04:00:00,87.63,87.63,87.63,87.63,87.63,0,0.0,0.0,False
2025-01-21 04:01:00,87.63,87.63,87.63,87.63,87.63,0,0.0,0.0,False
2025-01-21 04:02:00,87.63,87.63,87.63,87.63,87.63,0,0.0,0.0,False


In [15]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/VRT-2020-02-10.parquet")
filtered = bars.loc['2020-02-07 19:57:00':'2020-02-10 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2020-02-07 19:57:00,12.7949,12.7949,12.7949,12.7949,12.86,0,0.0,0.0,False
2020-02-07 19:58:00,12.7949,12.7949,12.7949,12.7949,12.86,0,0.0,0.0,False
2020-02-07 19:59:00,12.7949,12.7949,12.7949,12.7949,12.86,0,0.0,0.0,False
2020-02-10 04:00:00,13.1332,13.1332,13.1332,13.1332,13.20,0,0.0,0.0,False
2020-02-10 04:01:00,13.1332,13.1332,13.1332,13.1332,13.20,0,0.0,0.0,False
2020-02-10 04:02:00,13.1332,13.1332,13.1332,13.1332,13.20,0,0.0,0.0,False


In [16]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/MRSH-2026-01-14.parquet")
filtered = bars.loc['2026-01-13 19:57:00':'2026-01-14 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2026-01-13 19:57:00,180.8459,180.8459,180.8459,180.8459,182.7000,0,0.0,0.0,False
2026-01-13 19:58:00,180.8459,180.8459,180.8459,180.8459,182.7000,0,0.0,0.0,False
2026-01-13 19:59:00,180.8459,180.8459,180.8459,180.8459,182.7000,0,0.0,0.0,False
2026-01-14 04:00:00,180.9547,180.9547,180.9547,180.9547,183.2574,0,0.0,0.0,False
2026-01-14 04:01:00,180.9547,180.9547,180.9547,180.9547,183.2574,0,0.0,0.0,False
2026-01-14 04:02:00,180.9547,180.9547,180.9547,180.9547,183.2574,0,0.0,0.0,False


In [17]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/LOGC-2024-05-13.parquet")
filtered = bars.loc['2024-05-10 19:57:00':'2024-05-13 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2024-05-10 19:57:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False
2024-05-10 19:58:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False
2024-05-10 19:59:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False
2024-05-13 04:00:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False
2024-05-13 04:01:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False
2024-05-13 04:02:00,5.13,5.13,5.13,5.13,5.13,0,0.0,0.0,False


In [18]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/BKKT-2021-10-18.parquet")
filtered = bars.loc['2021-10-15 19:57:00':'2021-10-18 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2021-10-15 19:57:00,238.75,238.75,238.75,238.75,9.55,0,0.0,0.0,False
2021-10-15 19:58:00,238.75,238.75,238.75,238.75,9.55,0,0.0,0.0,False
2021-10-15 19:59:00,242.25,242.25,242.25,242.25,9.69,1938,200.0,2.0,True
2021-10-18 04:00:00,245.00,245.00,245.00,245.00,9.80,0,0.0,0.0,False
2021-10-18 04:01:00,245.00,245.00,245.00,245.00,9.80,0,0.0,0.0,False
2021-10-18 04:02:00,245.00,245.00,245.00,245.00,9.80,0,0.0,0.0,False


In [19]:
print(f"Last date in market_dates: {market_dates[-1]}")
print(f"end_data_from: {end_data_from}")
print(f"END_DATE: {END_DATE}")
print(f"Index of end_data_from: {market_dates.index(end_data_from)}")


Last date in market_dates: 2026-06-03
end_data_from: 2026-06-03
END_DATE: 2026-06-03
Index of end_data_from: 2512


In [20]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

1089


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
5616,META-2022-06-09,META,{'2022-06-09': 'FB'},"Meta Platforms, Inc. Class A Common Stock",True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1326801.0,BBG000MM2P62


In [21]:
# Check for Negative Prices again! Think I saw one!

# List of tickers with Negative prices
PROCESSED_PATH = "../data/polygon/processed/m1"
FILE_EXT = ".parquet"

for file in os.listdir(PROCESSED_PATH):
    if file.endswith(FILE_EXT):
        ticker = file.replace(FILE_EXT, "")
        file_path = os.path.join(PROCESSED_PATH, file)        
        
        df = pd.read_parquet(file_path)
        
        if (df['close'] < 0).any():
            print(f'{ticker} has negative prices!')

KDP-2018-07-10 has negative prices!
GEN-2016-06-06 has negative prices!
CARM-2023-03-08 has negative prices!


# Verify that all the Data Cleaning processes have been applied correctly! Especially for stocks with ticker changes. 

## Verify OHLC and Volume. Read parquet files to verify backfills, transactions, turnover, volume, etc.
Example Ticker Changes: FB/META, BITF/KEEL, XON/XOM, WTW/WW, MMC/MRSH, LB/VSCO, RTN/RTX, GEN (formerly symantec)


## Verify many tickers by Plotting their charts and comparing against Tradingview 

Example Ticker Changes: FB/META, BITF/KEEL, XON/XOM, WTW/WW, MMC/MRSH, LB/VSCO, RTN/RTX, GEN (formerly symantec)

Furthermore, several large-cap companies have completely changed the exchange they trade on rather than their ticker—such as T-Mobile, Honeywell, and Kraft Heinz—which transferred their primary listings from the NYSE to the Nasdaq.
TMUS, HON, KHC

T-Mobile: Changed from TMUS to T (or various tracker shifts) following their merger with Sprint.
TMUS/T


In [22]:
""" # Plots of Daily data

# tickers = ['META', 'KEEL', 'XOM', 'WW', 'MRSH', 'VSCO', 'RTX', 'GEN', 'HON', 'KHC', 'TMUS', 'T']
tickers = ['META', 'KEEL']


for ticker in tickers:
    all_adjusted_bars = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")

    # Resample to daily OHLC
    daily = all_adjusted_bars.resample('1D').agg({'open': 'first', 
                                    'high': 'max', 
                                    'low': 'min', 
                                    'close': 'last'})

    # Remove any rows where all OHLC columns are NaN (if any)
    daily.dropna(inplace=True)

    # Plot last 10,000 days
    mpf.plot(daily.tail(10000), type='candle', show_nontrading=False, warn_too_much_data=1000000) """

' # Plots of Daily data\n\n# tickers = [\'META\', \'KEEL\', \'XOM\', \'WW\', \'MRSH\', \'VSCO\', \'RTX\', \'GEN\', \'HON\', \'KHC\', \'TMUS\', \'T\']\ntickers = [\'META\', \'KEEL\']\n\n\nfor ticker in tickers:\n    all_adjusted_bars = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")\n\n    # Resample to daily OHLC\n    daily = all_adjusted_bars.resample(\'1D\').agg({\'open\': \'first\', \n                                    \'high\': \'max\', \n                                    \'low\': \'min\', \n                                    \'close\': \'last\'})\n\n    # Remove any rows where all OHLC columns are NaN (if any)\n    daily.dropna(inplace=True)\n\n    # Plot last 10,000 days\n    mpf.plot(daily.tail(10000), type=\'candle\', show_nontrading=False, warn_too_much_data=1000000) '

In [23]:
""" # Plots of 1-min data

# List of tickers (or just one)
tickers = ['META', 'KEEL', 'XOM', 'WW', 'MRSH', 'VSCO', 'RTX', 'GEN', 'HON', 'KHC', 'TMUS', 'T']

for ticker in tickers:
    # Read the 1-minute parquet file
    df = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")

    # Ensure the index is datetime (it usually is)
    # If not, set it: df.index = pd.to_datetime(df.index)

    # Drop any rows where all OHLC are NaN
    df.dropna(subset=['open', 'high', 'low', 'close'], how='all', inplace=True)

    # --- Plot the last N bars (e.g., 10000 minutes ≈ 12-13 trading days) ---
    # Adjust the number based on your screen/memory
    mpf.plot(df.tail(10000), 
             type='candle',          # or 'candle' for candlesticks
             volume=True,         # set to True if you have volume column
             show_nontrading=False,
             warn_too_much_data=1000000,
             title=f"{ticker} - 1-min OHLC",
             figsize=(12, 6)) """

' # Plots of 1-min data\n\n# List of tickers (or just one)\ntickers = [\'META\', \'KEEL\', \'XOM\', \'WW\', \'MRSH\', \'VSCO\', \'RTX\', \'GEN\', \'HON\', \'KHC\', \'TMUS\', \'T\']\n\nfor ticker in tickers:\n    # Read the 1-minute parquet file\n    df = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")\n\n    # Ensure the index is datetime (it usually is)\n    # If not, set it: df.index = pd.to_datetime(df.index)\n\n    # Drop any rows where all OHLC are NaN\n    df.dropna(subset=[\'open\', \'high\', \'low\', \'close\'], how=\'all\', inplace=True)\n\n    # --- Plot the last N bars (e.g., 10000 minutes ≈ 12-13 trading days) ---\n    # Adjust the number based on your screen/memory\n    mpf.plot(df.tail(10000), \n             type=\'candle\',          # or \'candle\' for candlesticks\n             volume=True,         # set to True if you have volume column\n             show_nontrading=False,\n             warn_too_much_data=1000000,\n             title=f"{ticker} - 1-min OHL

Tickers that were renamed multiple times:

In [24]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

95


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
19,AAMI-2025-01-02,AAMI,"{'2025-01-02': 'BSIG', '2018-03-26': 'OMAM'}",Acadian Asset Management Inc.,True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1748824.0,BBG00P2HLNY3
64,ABTC-2025-09-03,ABTC,"{'2025-09-03': 'GRYP', '2024-02-09': 'KERN'}",American Bitcoin Corp. Class A Common Stock,True,2019-06-18,2026-06-03,2019-06-18,2026-06-03,CS,1755953.0,BBG00MS8GZP1


In [26]:
assert False

AssertionError: 

# Data Cleaning is Finished!
Now we have multiple ticker lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. Data start/end dates added (in 03_tickers.ipynb). 
4. ETFs added.
5. Adjustments (Splits, Dividends) applied whilst keeping `close_original` for PIT Price.
6. Data Cleaned thoroughly (with "backfills" applied to all 1-min bars - especially useful for Pre-mkt bars) under 07_process.ipynb
7. Renamings (ticker changes) merged and stitched (FB/META, BITF/KEEL). tickers_definitive_with_renamings.csv

**Only the last should be used in backtesting.**

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

# 8.2 Updates
Rerun the file after setting END_DATE and updating the list of renamings.

# <ins>Explanations</ins>

# Renamings - others have abandoned renamings. Worth the effort? Astounding YES! 
# But only made possible by stockanalysis.com's curated "Ticker Changes" database! 
### Stockanalysis.com also moves Merger&Aquisitions to a separate database and that keeps the "Ticker Changes" database fairly clean for our TA-based strategies 
**stockanalysis.com's changes database is already pre-filtered.** It tracks situations where an old ticker had a named successor from the perspective of the original shareholders. Pure spin-offs don't appear as ticker changes in that DB — GEHC didn't show up as "GE renamed to GEHC." Spin-off children appear as new listings. What DOES appear is cases like MYL → VTRS, where MYL shareholders received VTRS shares and the ticker was retired. That's shareholder continuity by definition, which is exactly the stitching criterion.

The curation work has largely been done for you. The database being called "changes" rather than "M&A events" is semantically meaningful.

**Conclusion:**
- very small percentage of our universe is Grey area (cos pure spin-offs and most mergers/acquisitions are excluded from ticker changes DB). And the cost of not stitching is much higher!
- the remaining stitchings happen correctly via our code. 

### (if using Fundamentals, they will need to be cleaned too but we don't use Fundamentals). 

## ChatGPT 5.5
#### 1. Pure ticker/name changes → STITCH

Examples:

| Old  | New                              | Action |
| ---- | -------------------------------- | ------ |
| FB   | META                             | Stitch |
| MMC  | MRSH                             | Stitch |
| GOOG | GOOGL (if same class continuity) | Stitch |
| WTW  | WW                               | Stitch |
| SQ   | XYZ                              | Stitch |
| BRCM | AVGO                             | Stitch |
| BITF | KEEL                             | Stitch |
| BTC  | Treasuries                       | Stitch |
| Hype | Stocks                           | Stitch |

Characteristics:

* Same legal entity
* Same shareholders
* Same assets/business
* Same CUSIP/FIGI usually
* Exchange announces "ticker change"

For backtesting:

```
FB history
+
META history
=
one continuous META series
```

---

#### 2. Spin-offs → DO NOT STITCH

Examples:

| Parent | Spin-off |
| ------ | -------- |
| KFT    | MDLZ     |
| PYPL   | EBAY     |
| ABBV   | ABT      |
| GE     | GEHC     |
| GE     | GEV      |

Characteristics:

* New legal entity created
* Separate stock begins trading
* Shareholders receive new shares

For backtests:

Treat them as separate securities.

The parent gets a corporate action adjustment if you care about total return.

Example:

```
ABT ------>
          \
           ---> ABBV starts
```

Do NOT create:

```
ABT + ABBV
```

---

#### 3. Reverse mergers / SPACs → Usually STITCH

Example:

```
CCIV → LCID
IPOB → OPEN
DWAC → DJT
```

Although the business changes dramatically:

* the listed shell survives,
* shareholders continue owning the surviving entity.

For systematic equity research:

I stitch these.

Reason:

The security itself survives.

You could optionally flag:

```
identity_change = True
```

because fundamentals before/after are incomparable.

---

#### 4. Bankruptcy reorganizations → Usually BREAK

Examples:

```
GM → MTLQQ → new GM
AIG old → restructured AIG
```

If old equity is extinguished:

Do not stitch.

Old shareholders lost ownership.

New stock is a new investment.

---

#### 5. Mergers of equals / transformational mergers → GRAY AREA

This is where MYL/VTRS sits.

These are the hardest cases.

---

#### MYL → VTRS

Timeline:

```
MYL (Mylan)
   +
Pfizer's Upjohn business
        ↓
VTRS (Viatris)
```

Mechanics:

* Mylan legally survived.
* Mylan shareholders became majority owners (~57%).
* Pfizer shareholders received VTRS shares (~43%).
* MYL ticker disappeared.
* VTRS ticker began.

This was not merely:

```
MYL rebrands to VTRS
```

Nor was it:

```
MYL dies and VTRS is born from scratch.
```

It was a transformational merger.

---

#### What would I do?

For a professional security master:

### Security continuity:

#### YES

I would stitch:

```
MYL → VTRS
```

because:

* MYL shareholders retained ownership.
* MYL legal entity survived.
* The listing continued.

---

#### But...

I would attach a flag:

```python
event_type = "transformational_merger"
continuity = True
identity_changed = True
```

because economically:

Pre-2020 MYL and post-2020 VTRS are very different companies.

---

#### Why this matters for quant strategies

Suppose you're computing:

##### Technical signals

Examples:

* momentum
* moving averages
* volatility
* breakout signals

Then stitching is correct.

The market treated VTRS as the continuation of MYL.

You want:

```
2015 ... 2020(MYL) ... 2026(VTRS)
```

as one price history.

---

##### Fundamental signals

Examples:

* PE
* EV/EBITDA
* revenue growth
* ROE
* margins

Then stitching is dangerous.

Because:

```
2019 MYL earnings
≠
2021 VTRS earnings
```

The business composition changed massively.

You should either:

* reset fundamental histories, or
* use the `identity_changed` flag to invalidate trailing calculations across the event.

---

#### The framework I would implement

I'd classify events like this:

| Event Type                         | Stitch OHLCV? | Fundamental Continuity? |
| ---------------------------------- | ------------: | ----------------------: |
| Ticker/name change                 |           Yes |                     Yes |
| Spin-off                           |            No |                      No |
| Pure merger into another ticker    |            No |                      No |
| Reverse merger/SPAC                |           Yes |              Usually No |
| Transformational merger (MYL→VTRS) |           Yes |                      No |
| Bankruptcy/new equity              |            No |                      No |

---

If I were in your shoes — **retail quant, 5,000+ US stocks, OHLCV only, multi-week/multi-month swing holding periods, limited budget/time** — I would not manually classify anything.

I would use this rule:

> **Stitch every ticker change that your vendor identifies as a ticker change.**
>
> **Do not stitch anything else.**

That's it.

Seriously.

---

#### Why?

Because the alternatives are worse.

##### Option 1: Manual review

Impossible.

5000 events × even 5 minutes each = 400+ hours.

Not happening.

---

##### Option 2: Never stitch

This creates massive problems.

Suppose:

```
FB history:
2012–2021

META history:
2021–
```

Then your momentum strategy in late 2021 sees:

```
META age = 3 days
252-day momentum = unavailable
200-day MA = unavailable
ATR history = unavailable
```

You'd lose a huge amount of usable history.

Same for:

```
GOOG → GOOGL
TWTR → X (if it existed publicly)
WWAV → WTW
SQ → XYZ
BRCM → AVGO     
BITF → KEEL  
BTC → Treasuries       
Hype → Stocks 
```

For swing strategies this is very damaging.

---

##### Option 3: Stitch everything vendor labels as ticker changes

This is what I would do.

The few weird cases (MYL→VTRS, SPACs, reverse mergers) are rare.

Think about the numbers.

Suppose:

```
5000 ticker-change links

4900 = ordinary renamings
50   = weird mergers
30   = SPACs
20   = borderline cases
```

Even if all 100 weird ones are "wrong":

```
100 / 5000 = 2%
```

Now ask:

How many of those 100 weird cases will materially affect your strategy results?

Maybe:

```
20–30 names over 20 years.
```

Out of:

```
5000 stocks
× 5000 trading days
≈ 25 million stock-days
```

The effect is microscopic.

---

#### The important insight

For OHLCV swing systems:

The biggest source of bias is **not** occasionally stitching a weird merger.

The biggest source of bias is:

##### Broken histories.

Example:

Without stitching:

```
META momentum unavailable
META MA unavailable
XYZ momentum unavailable
```

**for months after every ticker change!**

**That introduces systematic distortions.**

---

If I were building **your exact setup**, I would absolutely continue using the StockAnalysis ticker-change database as the backbone of the stitching process.

The key difference is this:

> I would trust StockAnalysis for ordinary ticker changes, but I would not blindly stitch 100% of them.

I would add one extremely simple automated filter.

---

#### Why StockAnalysis is actually a very good choice

You've already discovered something that many retail quants eventually learn:

* Polygon's ticker mappings are incomplete.
* FIGI/CUSIP/PERMNO mappings are often expensive, incomplete, or hard to maintain.
* CRSP is ideal but prohibitively expensive.
* Vendor identifiers break more often than people assume.

Meanwhile:

StockAnalysis appears to have put significant effort into maintaining ticker histories because their entire website depends on users finding the "same stock" through ticker changes.

For a retail quant:

> A curated ticker-change database is often more practical than trying to reconstruct CRSP.

I think that's a perfectly reasonable decision.

---

#### The real risk

The problem isn't:

```text
FB → META
```

The problem is these rare events:

```text
MYL → VTRS
CCIV → LCID
IPOB → OPEN
DWAC → DJT
```

where:

* the ticker changed,
* but the underlying business changed dramatically.

---

#### But here's the important question:

##### Does your strategy care?

You trade:

* OHLCV only,
* swing horizons,
* multi-week/multi-month holds.

For those strategies:

##### Price continuity is much more important than business continuity.

Momentum doesn't know what Lipitor is.

Trend following doesn't care that Upjohn merged into Mylan.

ATR doesn't care whether Lucid used to be a SPAC.

---

#### Therefore, I would do this

##### Stage 1: Stitch everything from StockAnalysis

If StockAnalysis says:

```text
old_ticker → new_ticker
```

then:

```python
stitch = True
```

This preserves:

* moving averages,
* momentum,
* volatility estimates,
* breakout signals,
* relative strength calculations.

This is the biggest win.

---

#### Stage 2: Add a discontinuity check

This is the one thing I'd add.

Suppose:

```python
gap = abs(
    new_first_open / old_last_close - 1
)
```

Then:

##### If gap ≤ 50%

Accept automatically.

```python
stitch = True
flag = False
```

---

##### If gap > 50%

Still stitch...

but mark it:

```python
stitch = True
flag = True
```

For example:

| Event                  | Approx Gap |
| ---------------------- | ---------: |
| FB→META                |       1–5% |
| SQ→XYZ                 |      5–10% |
| MYL→VTRS               |    ~10–20% |
| SPAC mergers           |   100–500% |
| Extreme restructurings | Very large |

This will isolate the weird cases.

---

#### Why not break them?

Because your signals need history.

Imagine LCID.

If you break:

```text
CCIV history ends
LCID starts
```

then for months:

```text
252-day momentum = unavailable
200-day MA = unavailable
volatility regime = unavailable
```

That is probably more damaging than the fact that Lucid used to be a SPAC shell.

---

#### The one exception I WOULD make

If the first trading day after the change shows something insane:

```python
gap > 300%
```

I might break the chain.

Because that's often:

* bankruptcy/new equity,
* extreme reverse mergers,
* bizarre restructurings.

Those events are extremely rare.

Maybe:

```text
5–20 cases
out of 5,000 mappings.
```

You can even review those manually once.

---

#### So the exact algorithm I'd use

```python
for old, new in stockanalysis_changes:

    gap = abs(first_open[new] / last_close[old] - 1)

    if gap <= 3.0:      # 300%
        stitch(old, new)

    else:
        break_chain(old, new)
        log_review(old, new, gap)
```

That's it.

No PERMNO.

No FIGI.

No manual classification of 5,000 events.

No corporate-action detective work.

---

#### Would I personally use this for a serious retail swing system?

Yes.

In fact, if I had:

* Polygon OHLCV,
* StockAnalysis full ticker history,
* no access to CRSP,
* a focus on technical swing strategies,

this is almost exactly the solution I would deploy.

I think the biggest mistake would be striving for institutional-grade perfection and ending up with a fragile, half-finished pipeline.

A simple rule like:

> **"Stitch all StockAnalysis ticker changes, except for a handful of absurd discontinuities"**

gets you 95–99% of the benefit with perhaps 1% of the effort. For a lone retail quant, that's an excellent trade-off.

## Claude sonnet 4.6:
Given your specific constraints — OHLCV only, multi-week/multi-month holds, 5000 changes, solo operation — the answer is actually simple.

##### The Blanket Rule: Stitch Everything in the stockanalysis.com Ticker Changes DB

And here's why this is actually defensible, not just pragmatic laziness.

**stockanalysis.com's changes database is already pre-filtered.** It tracks situations where an old ticker had a named successor from the perspective of the original shareholders. Pure spin-offs don't appear as ticker changes in that DB — GEHC didn't show up as "GE renamed to GEHC." Spin-off children appear as new listings. What DOES appear is cases like MYL → VTRS, where MYL shareholders received VTRS shares and the ticker was retired. That's shareholder continuity by definition, which is exactly the stitching criterion.

The curation work has largely been done for you. The database being called "changes" rather than "M&A events" is semantically meaningful.

##### Why It's Fine for OHLCV-Only Mid-Term Strategies

The theoretical objections from the previous discussion mostly don't apply to you:

- **Fundamental breaks** — irrelevant, you don't use them
- **Shares outstanding explosion** — irrelevant for price-based signals
- **Market cap misclassification** — not a factor in your signals

What matters for your strategies is whether momentum, moving averages, ATR, and similar lookback indicators are computed on a coherent price series. A stitched MYL/VTRS series gives you exactly that. An unstitched series gives you VTRS appearing as a 2020 IPO with zero lookback history, which is actively harmful to your signals.

**The cost of wrongly stitching an edge case: minor.** One slightly discontinuous price level at the join point.

**The cost of wrongly not stitching: severe.** You lose all history on the new ticker, generate false breakout signals on day one, can't compute any lookback window indicators, and introduce survivorship bias artifacts.

For multi-week holds the noise from an edge case stitch is negligible.

##### The One Automated Sanity Check Worth Adding

You don't need to manually review 5000 records, but add one programmatic check:

```python
# At each stitch point, compare last close of old ticker 
# to first open of new ticker
price_gap_pct = abs(new_open - old_close) / old_close

STITCH_GAP_THRESHOLD = 0.20  # 20%

if price_gap_pct > STITCH_GAP_THRESHOLD:
    flag_for_review(old_ticker, new_ticker, price_gap_pct)
```

A clean rename or 1:1 merger will produce a gap of near zero. A non-1:1 exchange ratio merger (e.g., you got 0.7 new shares per old share) will produce a price-level discontinuity that this catches automatically. Even then, your flagged list will be a small fraction of 5000, and you can decide case by case whether to stitch or not.

Everything under the threshold: stitch without further review.

##### The Only Real Exception Class to Keep in Mind

**Bankruptcy emergences** — if old ticker was OTC/pink sheets trading at pennies and new ticker opened at $10+, that gap check will catch it. These sometimes appear in the stockanalysis changes DB because technically a new ticker was issued to (some of) the old shareholders. Do not stitch these. But the price gap filter handles it automatically.

---

**Summary:** Trust the stockanalysis.com database's curation, stitch everything, add the price gap sanity check to auto-flag outliers, and move on. For a pure OHLCV swing trader this is the right pragmatic call and it won't meaningfully hurt your backtest quality. The hour you'd spend building a sophisticated classification system is better spent on strategy research.

# Design Spec:

Summary:
- Adjustments (Splits and Split-adjusted Cash Dividends).
    - Get Adjustment DB from Polygon.
    - Clean-up Adjustments. 
        - Split-adjust Cash Dividends. 
    - Apply cleaned-up Adjustments (Both Splits and Split-adjusted Cash Dividends) to:
        - OLD ticker's Adjustments to OLD ticker's OHLCV.
        - NEW ticker's Adjustments to NEW ticker's OHLCV.
- Ticker Changes
    - Get Database of Ticker_Changes from stockanalysis.com
    - Filter out the companies that are structurally different Spin-offs and don't need the Stitching applied to them. (CODE NEEDS TO BE CHANGED TO ACCEPT FILES RATHER THAN 1 MERGED FILE!)
    - ONLY Split-adjust and apply the following steps to companies that are structurally the same! 
        - Sometimes, Split info is on the NEW Ticker's start date. Grab that and apply to OLD ticker since NEW ticker will obviously open at the NEW naturally-adjusted price. 
        - Apply an "Adjustment_factor" to apply ALL the adjustments on the NEW ticker to the OLD ticker.
        - Stitch OLD ticker and NEW ticker data and get rid of old files. 
- CLEAN DATA!


Details:
1. The "exclusive" start‑date filter in the new ticker’s adjustment (prevents start‑date splits from being applied to NEW’s own data). **(ORIGINAL)**
	- split_adjust() function has `adjustments.index > trading_dates_ticker[0]` to make sure that we don't apply any splits that have the same date as the first trading date. This is because this prevents situations where a NEW_RENAMED ticker has a Spilt with the same date as the start_date of this ticker and yet the stock will obviously start trading at the NEW_Price so we can NOT apply this Split adjustment to the start_date or else we will get duplication. Example: OLD ticker price was $100 and NEW_RENAMED ticker starts trading at $20. 
	- A BRAND_NEW ticker won't be affected by this obviously! 
2. The “annoying” block applies those start‑date splits (and dividends) directly to the old ticker. **(ORIGINAL)**
3. Use the first‑bar `close / close_original` as the `adjustment_factor` to propagate ALL the NEW_RENAMED ticker adjustments to the OLD ticker. **(ORIGINAL)**
	- Use the first‑bar, NOT the last-bar since both `close` and `close_original` will be same for the last-bar as we have been doing backward adjustment to retain the current price, i.e. current price is equal to the actual market price currently. 
4. Repeat for Dividends. **(ADDED)**
	- Additionally, Dividends need to be adjusted for splits so make sure you add that code too. **(ADDED. Huge oversight on the Author' part - likely polygon changed things).** 
5. Verifications **(ADDED)**
6. Inferred Renamings **(ADDED)** (Verify that this is being propogated correctly).
7. Negative Prices and whitelisting/blacklisting. **(IMPROVED)**
    - Added safety checks for edge cases that are genuine but give negative prices (cos made by company in deep trouble and readying for bankruptcy). We only found one such ticker (GEN-2016-06-06) but we added safety checks and structure around this so we deal with any issues graciously in the LIVE environment and in a pre-calculated way.
7. NOT stitch structurally different Spin-offs! (see relevant section) **(ADDED)**
    - See if we need to manually whitelist or blacklist them, or can be automated if only a few.
    - MUST use Volume as your guide, especially for big gaps so we can decide whether to include them as part of our lookbacks or not.
8. Dev's other improvements:
    - Add "transactions" data. Alpha!
    - Include "Volume" back. It's useful for intraday calculations like VWAP for an individual stock - even if it's not useful for cross-sectional comparisons. 
9. Concretum's tests. **(ADDED)**
10. VERIFY whether Volume, transactions, etc. needs adjusting too?

Keep renamings and process files separate. We don't want to drift too much from the original given it's most of our code.



# Concept

### "Tickers and Company names are arbitrary, re-useable things." And we can't use composite_FIGI or CIK as unique identifiers (see Introduction). So this is the only method that works! 
This notebook is designed to handle one of the most complex challenges in building a stock price database: properly accounting for ticker symbol changes (renamings) over time. 
Because the Polygon.io API stores data by ticker symbol, which can change when a company rebrands or is acquired, the goal is to create a unified, historical record that links all past symbols for a company to its current one, ensuring a continuous price history without gaps or data loss.

### 💡 Problem: Ticker-Centric Data (FB ticker data stays with FB ticker and doesn't get passed to META on ticker change. BITF/KEEL, etc.)
A key challenge with Polygon.io data (and others like Norgate) is that it's "ticker-centric". If a company changes its ticker (e.g., from `A` to `B`), the historical price data for `A` and the new data for `B` are stored separately. This notebook aims to solve this by creating a mapping that tells you, for any given date, which ticker symbol to use for a specific company. As the notebook's author notes, without this, a backtest might incorrectly assume a stock stopped trading when it simply changed its ticker.

### 🔀 Where the Data Comes From: `stockanalysis.com` (Cheap/free Solution = free trial or $10 one-off, and for ongoing updates the site gives last 12 months ticker changes for FREE under the public "Recent ticker changes" public endpoint with no login required!)
The notebook explicitly states that building this mapping directly from Polygon.io's own `Ticker Events` endpoint is problematic and doesn't work for delisted companies. The author attempted an alternative method of inferring changes but found it "very messy".

The chosen solution is to source ticker change data from [**stockanalysis.com**](https://stockanalysis.com/actions/changes/), which requires a subscription (noted as costing $10/month and having a free first month). The user is instructed to manually download these CSV files for each year and place them in the folder `stockanalysis/raw/ticker_changes/`.

### ⚙️ What This Notebook Does (A Multi-Step Process)
The notebook processes the data from `stockanalysis.com` and integrates it with the main ticker list to produce a definitive master record.

1.  **Aggregate Data**: It reads all the manually downloaded CSV files from the `ticker_changes` folder and combines them into a single DataFrame.
2.  **Process & Map Changes**: The core logic involves a loop that goes through the existing ticker list and, for each ticker, looks for a change record on its "end date." If a change is found, the notebook:
    *   Merges the stock's data record with its new ticker symbol.
    *   Updates the main ticker list (`tickers_v4.csv`) to reflect the current symbol.
    *   Logs the historical change in a new column called `tickers_old`, which stores a list of `(date_of_change, old_ticker)` pairs. This creates a full audit trail.
3.  **Create the Final Master List**: The final output is a new file, `tickers_v5.csv`. This file serves as the definitive ticker list for the entire database, ensuring that every company's price history can be traced correctly through all its symbol changes.